In [ ]:
# ==============================================================================
# Reproducibility-Oriented Earthquake Dataset Preprocessing Pipeline
# ==============================================================================
# This implementation is designed to ensure:
# - deterministic preprocessing,
# - dataset fidelity preservation,
# - methodological transparency,
# NumPy random seed is fixed to ensure deterministic behavior
# in any future stochastic extensions of the preprocessing pipeline.
# ==============================================================================

import pandas as pd
import numpy as np
import os

# ==============================================================================
# Global Reproducibility Configuration
# ==============================================================================

np.random.seed(42)

# ==============================================================================
# Step 1: Data Acquisition
# ==============================================================================
# Relative paths enable execution across different systems.

raw_data_path = 'original_dataset.csv'

print(f"Loading raw dataset from '{raw_data_path}'...")

df = pd.read_csv(raw_data_path)

# ==============================================================================
# Step 2: Temporal Standardization
# ==============================================================================
# Temporal attributes are generated only if they are not already present.

if 'time' in df.columns and 'Year' not in df.columns:

    print("Performing temporal standardization...")

    df['time'] = pd.to_datetime(df['time'], utc=True)

    df['Year'] = df['time'].dt.year
    df['Month'] = df['time'].dt.month
    df['Day'] = df['time'].dt.day
    df['Hour'] = df['time'].dt.hour
    df['Min'] = df['time'].dt.minute
    df['Sec'] = df['time'].dt.second

# ==============================================================================
# Step 3: Deterministic Magnitude Severity Label Construction
# ==============================================================================
# Magnitude severity labels are generated only if absent.

if 'mag class' not in df.columns:

    print("Generating deterministic magnitude severity labels...")

    def categorize_magnitude(mag):

        if mag < 5.0:
            return 'light'

        elif 5.0 <= mag < 6.0:
            return 'moderate'

        elif 6.0 <= mag < 7.0:
            return 'Strong'

        else:
            return 'Major'

    df['mag class'] = df['mag'].apply(categorize_magnitude)

# ==============================================================================
# Step 4: Feature Alignment
# ==============================================================================
# Only experimentally relevant features are retained.

columns_to_keep = [
    'Day',
    'Month',
    'Year',
    'Hour',
    'Min',
    'Sec',
    'latitude',
    'longitude',
    'depth',
    'magType',
    'gap',
    'rms',
    'horizontalError',
    'depthError',
    'magError',
    'mag',
    'mag class'
]

df_aligned = df[
    [col for col in columns_to_keep if col in df.columns]
].copy()

print("\nSelected Features:")
print(df_aligned.columns.tolist())

# ==============================================================================
# Step 5: Missing Value Handling
# ==============================================================================
# Rows with missing target magnitude values are removed.

print("\nExecuting missing value handling pipeline...")

df_aligned.dropna(subset=['mag'], inplace=True)

# ------------------------------------------------------------------------------
# Separate Numerical and Categorical Features
# ------------------------------------------------------------------------------

num_cols = df_aligned.select_dtypes(include=[np.number]).columns.tolist()

cat_cols = df_aligned.select_dtypes(include=['object']).columns.tolist()

# ------------------------------------------------------------------------------
# Exclude Temporal Attributes from Numerical Imputation
# ------------------------------------------------------------------------------
# Temporal attributes are excluded to avoid synthetic timestamp generation.

temporal_cols = ['Day', 'Month', 'Year', 'Hour', 'Min', 'Sec']

num_cols = [
    col for col in num_cols
    if col not in temporal_cols
]

# ------------------------------------------------------------------------------
# Numerical Missing Value Imputation (Median)
# ------------------------------------------------------------------------------

print("Applying median imputation to numerical features...")

df_aligned[num_cols] = df_aligned[num_cols].fillna(
    df_aligned[num_cols].median()
)

# ------------------------------------------------------------------------------
# Categorical Missing Value Imputation (Mode)
# ------------------------------------------------------------------------------

print("Applying mode imputation to categorical features...")

for col in cat_cols:

    mode_values = df_aligned[col].mode()

    if not mode_values.empty:

        df_aligned[col] = df_aligned[col].fillna(
            mode_values[0]
        )

# ==============================================================================
# Step 6: Dataset Fidelity Preservation
# ==============================================================================
# Outlier clipping is intentionally avoided to preserve
# the original statistical distribution of seismic parameters
# and maintain dataset fidelity for reproducibility.

print("No outlier clipping applied.")

# ==============================================================================
# Step 7: Deterministic Chronological Ordering
# ==============================================================================
# Deterministic chronological ordering is applied to ensure
# reproducible row alignment across exported dataset versions.

sorting_columns = [
    col for col in
    ['Year', 'Month', 'Day', 'Hour', 'Min', 'Sec']
    if col in df_aligned.columns
]

if len(sorting_columns) > 0:

    print("Applying deterministic chronological ordering...")

    df_aligned = df_aligned.sort_values(
        by=sorting_columns
    ).reset_index(drop=True)

# ==============================================================================
# Step 8: Final Dataset Export
# ==============================================================================

output_directory = 'data'

os.makedirs(output_directory, exist_ok=True)

output_path = os.path.join(
    output_directory,
    'My dataset with class and without missing values.csv'
)

print(f"\nExporting harmonized dataset to:\n{output_path}")

df_aligned.to_csv(output_path, index=False)

# ==============================================================================
# Step 9: Final Dataset Validation Summary
# ==============================================================================

print("\n========================================")
print("FINAL DATASET INFORMATION")
print("========================================")

print(df_aligned.info())

print("\n========================================")
print("MISSING VALUES SUMMARY")
print("========================================")

print(df_aligned.isnull().sum())

print("\n========================================")
print("FINAL DATASET SHAPE")
print("========================================")

print(df_aligned.shape)

print("\n========================================")
print("FIRST FIVE ROWS")
print("========================================")

print(df_aligned.head())

print("\n========================================")
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("========================================")

Loading raw dataset from 'original_dataset.csv'...

Selected Features:
['Day', 'Month', 'Year', 'Hour', 'Min', 'Sec', 'latitude', 'longitude', 'depth', 'magType', 'gap', 'rms', 'horizontalError', 'depthError', 'magError', 'mag', 'mag class']

Executing missing value handling pipeline...
Applying median imputation to numerical features...
Applying mode imputation to categorical features...
No outlier clipping applied.
Applying deterministic chronological ordering...

Exporting harmonized dataset to:
data/My dataset with class and without missing values.csv

FINAL DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19402 entries, 0 to 19401
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Day              19402 non-null  int64  
 1   Month            19402 non-null  int64  
 2   Year             19402 non-null  int64  
 3   Hour             19402 non-null  int64  
 4   Min              1940